In [1]:
import pandas as pd
import numpy as np

### Capacity Aging Linear Approximation

The capacity in energy storage decrease over time. It decreases with aging, but also decreases with each performed cycle. What I want to do is to approximate this phenomena by a model, that can be injected into MILP optimization formulation.

Goal is to find formula for $cap_t$ - how the capacity of a storage change in time.

The idea is simple, for a given timestamp, I need to compute the $\Delta cap_t = cap_{t+1} - cap_t$. We can decompose it into $\Delta cap_t = \Delta cap_{age, t} + \Delta cap_{cycle, t}$.

#### Aging degradation
Aging process is kinda simple, and for now I would like to just model it, as a constant relative decrease over each year $\tau$ (for example, if $tau=0.01$, then we have a 1\%$ decrease in each year), so

$$
\Delta cap_{age, t} = cap_{t_0}\cdot \# y \cdot \tau.
$$

#### Cycle degradation
A cycle degradation is much harder topic, since each cycle decrease the capacity differently, cycles that are deeper decrease it more. Also, cycles at the beginning of the life of the storage, decrease it more, than the same cycles at the later stages.
In general, the cycle degradation depends on $dod$ (depth of discharge) of a cycle, and time in which the cycle was performed.

It is known, that the $\Delta cap_{cycle, t}$ is proportional to $(DOD_t)^{\gamma}$, where $\gamma\in[1.1, 1.3]$ (more or less). So the problem is, that it is not a linear function.

To formulate the model for a storage capacity cycle degradation, we need first to assume some level of capacity degradation at which we will decide, that the storage will stop operate. Let's call it $cap_{EOL}$ (end-of-life capacity).
Usually it is assumed to be around $80\%$ of the initial storage capacity.

Then we can assume, that the total energy throughput (in the storage lifetime) can be approximated as $T_{life} = N\cdot \frac{cap_{t_0} + cap_{EOL}}{2}$, where $N$ is the total number of cycles, that storage can perform.
This is an approximation and assumes linear degradation (which is not true, but this is the first version, so it is acceptable at this point).

Let's define a coefficient $\alpha = \frac{\Delta cap_{cycle, EOL}}{T_{EOL}}$ - the ratio of total capacity degradation divided by total energy throughput. This tells us, what amount of capacity (approximately) will be lost from a one unit of energy throughput.

Now, we can start by introducing some segmentation of the $[0,1]$ interval - creating capacity ranges. Let $s_i$ will be a vector of numbers from $[0,1]$, where $s_j < s_{j+1}$ and (why - it will be clear in a moment) for each $j$ we have that $P\cdot dt < cap_t\cdot (s_j - s_{j-1})$.
In other words, that in one unit of time ($dt$ is the time resolution, for example $0.25$ for $15$-minutes resolution) storage can not go from $soc_t = cap_t\cdot s_j$ to $soc_{t+1} > cap_t\cdot s_{j+1}$ in one timestep. We also postulate, that $s_0 = 0$ and $s_1 = 1$..

Now, let's decompose (for each $t$) $soc_t = \sum_j soc_{t,j}$ (the $j$ index is the same as for $s_j$). Next
$$
soc_{t,j} \le (s_{j+1} - s_j)\cdot cap_t
$$
$$
e_{t,j} = \sum_{j=0}^k soc_{t,j}
$$
$$
e_{j,t} \le e_{t, j+1}
$$

in that way we are sure, that if $soc_{t,j} > 0$, then all $soc_{t,l}$ are at the maximum level for all $l<j$. This will be very important property for us later.

Now, we need to define the delta state-of-charge as $\Delta soc_{t,j} \ge 0$ and $\Delta soc_{t, j} \ge soc_{t+1,j} - soc_{t,j}$.

The function $F(x) = x^{1.2}$ is non-linear, which is not a good information for us. We want to be able to somehow approximate the $F(DOD)\cdot soc_{max}$. We can visualize a cycle as "path upward" on the $soc(t)$ graph (where $soc(t)$ is a continuous function). When we reach the
top (the local maximum), we are on the "top" of a cycle. So the idea is this.

Let's define the following $w_j = F(s_j) - F(s_{j-1})$. The cycle degradation formula is this:

$$
\Delta cap_{cycle, t} = \alpha\cdot \sum_j w_j\cdot\Delta soc_{t,j}.
$$

Now, let's say we want to compute a degradation for a cycle, for which $soc_max = 0.6\cdot cap$. We can see, that no matter what path will lead us to the given $soc_max$, the final degradation equals $\alpha\cdot soc_max\cdot F(s_j)$, where $s_j$ is the end-point of the capacity
range in which $soc_max$ lies. This approach is much better than the industry standard used in the energy storage system models in the context of energy markets (models I find in the literature).

For most accurate approximation model, we will choose for all $j$ that $s_j - s_{j-1} = (dt + \epsilon)\cdot\frac{P}{cap}$. In that way we will enforce, that storage can not go from $soc_t = cap_t\cdot s_j$ to $soc_{t+1} > cap_t\cdot s_{j+1}$ in one timestep, and the number of
the approximation nodes will be maximal.

In [ ]:
def F(x) -> float:
    """
    Penalization / degradation factor applied to the DOD_t
    """
    return x ** 1.2

